# Embedding

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
type(v1), len(v1)

(numpy.ndarray, 384)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv), dv.dot(v1)

(np.float32(0.32332402), np.float32(0.32332402))

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.01973048)

# Embedding FAQ dataset

In [8]:
!ls ../dataset/faq 

ai-dev-tools-zoomcamp.json            machine-learning-zoomcamp.json
data-engineering-zoomcamp.json        mlops-zoomcamp.json
llm-zoomcamp.json                     stock-markets-analytics-zoomcamp.json


In [9]:
import json
from itertools import chain
from pathlib import Path

def loader(fp: list[Path]):
    with open(fp) as f:
        return json.load(f)

def load_dir(directory: Path) -> list[Path]:
    dir_path = Path(directory)
    files = [f for f in dir_path.iterdir() if f.is_file()]
    return files

def load_all_faqs(directory: Path):
    files = load_dir(directory)
    return list(chain.from_iterable(loader(f) for f in files))

In [10]:
faq_dir = Path("../dataset/faq")

faqs = load_all_faqs(faq_dir)

In [11]:
def unwrap(faq):
    return faq['question'] + ' ' + faq['answer']

In [12]:
faqs_unwrapped = list(map(unwrap, faqs))

In [13]:
len(faqs_unwrapped)

1349

In [14]:
from tqdm.auto import tqdm

In [15]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(faqs_unwrapped), batch_size)):
    batch = faqs_unwrapped[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1349

In [16]:
import numpy as np
X = np.array(vectors)

In [17]:
X

array([[-0.09314344, -0.1190973 , -0.08293633, ...,  0.01766491,
        -0.02965045, -0.02670314],
       [-0.05731956, -0.06769245, -0.06630197, ..., -0.01994153,
        -0.10189418,  0.01502755],
       [-0.09424276, -0.01793805, -0.00352702, ...,  0.10713081,
         0.00956119, -0.00851306],
       ...,
       [ 0.05554137,  0.02403271,  0.04465542, ...,  0.05873398,
         0.02059001, -0.01331506],
       [-0.05654018,  0.04030367, -0.00864579, ..., -0.01014855,
         0.02993989, -0.00998502],
       [ 0.00182531, -0.01129563, -0.1215924 , ...,  0.00073081,
         0.04048846, -0.03882811]], shape=(1349, 384), dtype=float32)

In [18]:
def vector_search(query, num_results=5):
    v = model.encode(query)
    scores = X.dot(v)
    top5 = np.argsort(-scores)[:5]
    return [{idx: faqs_unwrapped[idx], "score": scores[idx]} for idx in top5]

In [19]:
query = "Can I still enroll the course after the start date?"

In [20]:
vector_search(query)

[{np.int64(947): "Course: Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'score': np.float32(0.7356125)},
 {np.int64(599): "Course - Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'score': np.float32(0.7299192)},
 {np.int64(154): 'The course has already started. Can I still join it? Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a cer

In [21]:
q2 = "Can I get the certificate if I am a self-paced learner?"
vector_search(q2)

[{np.int64(959): "Certificate - Can I follow the course in a self-paced mode and get a certificate? No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
  'score': np.float32(0.78701687)},
 {np.int64(602): "Certificate - Can I follow the course in a self-paced mode and get a certificate? No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.",
  'score': np.float32(0.78701687)},
 {np.int64(46): 'Certificate: Can I follow the course in a self-paced mode and get a certificate? No, you can only get a certificate if you finish the course wi

# RAGVector

In [22]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, faqs)

In [24]:
import os

from dotenv import load_dotenv
from openai import OpenAI

from libs.ingest import load_all_faqs, build_index
from libs.rag_vector import RAGVector

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/",
)

index = build_index(faqs)
assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [25]:
assistant.rag("the program has already begun, can I still sign up?")

"# Yes, you can still sign up!\n\nAccording to the course information, even though the program has already begun, you can still join. However, there's one important condition:\n\n**If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.**\n\nYou can start learning and submitting homework right away without waiting for a confirmation email. Registration is not strictly required to participate—it's mainly used to gauge interest before the course starts."